In [23]:
import sys
sys.path.insert(0, '../')

import re
import numpy as np
import pandas as pd

In [38]:
def _str_to_interval(s):
    m = re.match(r'\[([-\d.]+),\s*([-\d.inf]+)\)', str(s))
    if m:
        return pd.Interval(float(m.group(1)), float(m.group(2)), closed='left')
    return s

In [80]:
def _collapse_to_health_bracket(iv):
    """Map a 5-year age bracket to the matching health-survey bracket and the
    fraction of that bracket which falls inside it. Returns (target, weight)
    or None for brackets below 18 that have no health counterpart."""
    if iv == pd.Interval(15.0, 19.0, closed='left'):
        # health starts at 18, so only ages 18-19 of the [15, 20) span count
        return pd.Interval(18.0, 34.0, closed='left'), 2 / 5
    left = iv.left
    if left < 18:
        return None
    if left < 34:
        return pd.Interval(18.0, 34.0, closed='left'), 1.0
    if left < 49:
        return pd.Interval(35.0, 49.0, closed='left'), 1.0
    if left < 64:
        return pd.Interval(50.0, 64.0, closed='left'), 1.0
    return pd.Interval(65.0, np.inf, closed='left'), 1.0

In [81]:
data_age = pd.read_csv('../data/bayesian_network/age-fixed.csv')
data_health = pd.read_csv('../data/bayesian_network/health-fixed.csv')

data_age = data_age.rename(columns = {'gender': 'sex'})
data_age = data_age.set_index(['sex', 'age_group'])
data_age.index = data_age.index.set_levels(
    data_age.index.levels[data_age.index.names.index('age_group')].map(_str_to_interval),
    level='age_group'
)
data_age = data_age.sort_index()

data_health = data_health.set_index(['sex', 'age_group'])
data_health.index = data_health.index.set_levels(
    data_health.index.levels[data_health.index.names.index('age_group')].map(_str_to_interval),
    level='age_group'
)
data_health = data_health.sort_index()

data_health = data_health.rename(
    columns = {
        'Body mass index, adjusted self-reported, adult (18 years and over), obese 15 16 17 18 19 20': 'Obese',
        'High blood pressure 21 22': 'High blood pressure',
        'Current smoker, daily or occasional 23 24 25 26 27': 'Current smoker',
        'Influenza immunization in the past 12 months 28 29': 'Recently vaccinated'
    }
)

In [84]:
_mapped = [_collapse_to_health_bracket(iv) for iv in data_age.index.get_level_values('age_group')]
_keep = [m is not None for m in _mapped]

_weights = np.array([m[1] for m in _mapped if m is not None])
_target_age = [m[0] for m in _mapped if m is not None]
_sex = data_age.index.get_level_values('sex')[_keep]

data_age = (
    data_age[_keep]
    .mul(_weights, axis=0)
    .set_axis(pd.MultiIndex.from_arrays([_sex, _target_age], names=['sex', 'age_group']))
    .groupby(level=['sex', 'age_group'])
    .sum()
    .round()
    .astype(int)
    .sort_index()
)

---------------------------

In [83]:
data_health

Obese  High blood pressure  Current smoker  \
sex     age_group                                                    
Females [18.0, 35.0)   974800                84600          330200   
        [35.0, 50.0)  1132100               308100          425100   
        [50.0, 65.0)  1185800               913300          572000   
        [65.0, inf)   1045400              1700900          304300   
Males   [18.0, 35.0)   955200               122900          562800   
        [35.0, 50.0)  1258100               442000          626900   
        [50.0, 65.0)  1284600              1130600          649900   
        [65.0, inf)    869500              1440800          325600   

                      Recently vaccinated  
sex     age_group                          
Females [18.0, 35.0)              1037900  
        [35.0, 50.0)              1181800  
        [50.0, 65.0)              1454300  
        [65.0, inf)               2284600  
Males   [18.0, 35.0)               647100  
        [35.0, 50.0)               812500  
        [50.0, 65.0)              1215600  
        [65.0, inf)               1932500

In [85]:
data_age

Alberta  British Columbia  Manitoba  New Brunswick  \
sex    age_group                                                          
Men+   [18.0, 34.0)   474585            588240    162911          71530   
       [35.0, 49.0)   503300            533696    137407          74305   
       [50.0, 64.0)   415099            515352    126363          88043   
       [65.0, inf)    315111            492079    108260          86280   
Women+ [18.0, 34.0)   456912            565372    145038          68023   
       [35.0, 49.0)   488296            536291    132760          76101   
       [50.0, 64.0)   409435            543194    127076          90225   
       [65.0, inf)    353094            562095    127873          98827   

                     Newfoundland and Labrador  Nova Scotia  Ontario  \
sex    age_group                                                       
Men+   [18.0, 34.0)                      44549       103160  1710779   
       [35.0, 49.0)                      47901        91460  1442995   
       [50.0, 64.0)                      61573       107741  1477991   
       [65.0, inf)                       59534       104012  1258456   
Women+ [18.0, 34.0)                      41757        96751  1578289   
       [35.0, 49.0)                      49359        94334  1472815   
       [50.0, 64.0)                      63662       113318  1537821   
       [65.0, inf)                       67441       121296  1504979   

                     Prince Edward Island  Quebec  Saskatchewan  \
sex    age_group                                                  
Men+   [18.0, 34.0)                 17396  830564        119413   
       [35.0, 49.0)                 14549  866112        118319   
       [50.0, 64.0)                 16633  880428        106313   
       [65.0, inf)                  15789  827768         95575   
Women+ [18.0, 34.0)                 16810  784817        111351   
       [35.0, 49.0)                 15636  836911        114405   
       [50.0, 64.0)                 17613  868198        105578   
       [65.0, inf)                  18734  959871        109072   

                     Canada (except provinces)  
sex    age_group                                
Men+   [18.0, 34.0)                    4123127  
       [35.0, 49.0)                    3830044  
       [50.0, 64.0)                    3795536  
       [65.0, inf)                     3362864  
Women+ [18.0, 34.0)                    3865120  
       [35.0, 49.0)                    3816908  
       [50.0, 64.0)                    3876120  
       [65.0, inf)                     3923282